In [ ]:
#setup
import math
import sys
from collections import Counter
from fractions import Fraction
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)
pd.options.display.precision = 4
rng = np.random.default_rng(42)
print(sys.executable)

In [ ]:
#sample space, listed completely
two_dice = list(product(range(1, 7), repeat=2))              # every (first, second) pair
print(len(two_dice), two_dice[:6])

def prob(event, space):
    """P(event) when every outcome in the space is equally likely."""
    favourable = [o for o in space if event(o)]
    return Fraction(len(favourable), len(space))

print("sum is 7        :", prob(lambda o: sum(o) == 7, two_dice))
print("both sixes      :", prob(lambda o: o == (6, 6), two_dice))
print("at least one six:", prob(lambda o: 6 in o, two_dice))
print("sum is 10 or more:", prob(lambda o: sum(o) >= 10, two_dice))
print("doubles         :", prob(lambda o: o[0] == o[1], two_dice))

In [ ]:
#the distribution of the sum of two dice
sums = Counter(sum(o) for o in two_dice)
print(sorted(sums.items()))

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.bar(list(sums.keys()), [c / 36 for c in sums.values()], color="lightsteelblue", edgecolor="white")
ax.set_xlabel("sum of two dice")
ax.set_ylabel("probability")
ax.set_xticks(range(2, 13))
plt.show()
print("probabilities add up to:", sum(Fraction(c, 36) for c in sums.values()))

In [ ]:
#empirical probability and the law of large numbers
flips = rng.integers(0, 2, size=10_000)                      # 0 = tails, 1 = heads (the upper bound 2 is NOT included)
heads = np.cumsum(flips)
n = np.arange(1, len(flips) + 1)
running = heads / n

for k in (10, 100, 1000, 10_000):
    print(f"after {k:>6} flips: proportion of heads = {running[k - 1]:.4f}, "
          f"heads minus tails = {abs(2 * heads[k - 1] - k)}")

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(n, running)
ax.axhline(0.5, color="tomato", ls="--")
ax.set_xscale("log")
ax.set_xlabel("number of flips")
ax.set_ylabel("proportion of heads")
plt.show()

In [ ]:
#he gambler’s fallacy
big = rng.integers(0, 2, size=1_000_000)
five_heads = np.convolve(big, np.ones(5, dtype=int), mode="valid") == 5     # True where 5 heads in a row end
after = np.flatnonzero(five_heads[:-1]) + 5                                 # the flip right after each streak
print(f"{len(after):,} streaks of 5 heads; the next flip is heads {big[after].mean():.3f} of the time")

In [ ]:
#a deck of cards
ranks = ["A"] + [str(r) for r in range(2, 11)] + ["J", "Q", "K"]
suits = ["hearts", "diamonds", "clubs", "spades"]
deck = list(product(ranks, suits))
print(len(deck))

king = lambda c: c[0] == "K"
heart = lambda c: c[1] == "hearts"
print("P(king)           :", prob(king, deck))
print("P(heart)          :", prob(heart, deck))
print("P(king and heart) :", prob(lambda c: king(c) and heart(c), deck))
print("P(king or heart)  :", prob(lambda c: king(c) or heart(c), deck))
print("addition rule     :", prob(king, deck) + prob(heart, deck) - prob(lambda c: king(c) and heart(c), deck))
print("P(king or queen)  :", prob(lambda c: c[0] in ("K", "Q"), deck), "(mutually exclusive, so just add)")

In [ ]:
#the complement trick, and the birthday problem
# "At least one" problems are easier through the complement
print("at least one head in 5 flips :", 1 - Fraction(1, 2) ** 5)
print("at least one six in 4 rolls  :", round(1 - (5 / 6) ** 4, 4))
print("at least one double six in 24 rolls of two dice:", round(1 - (35 / 36) ** 24, 4))

def p_shared_birthday(n):
    p_all_different = 1.0
    for i in range(n):
        p_all_different *= (365 - i) / 365
    return 1 - p_all_different

for people in (10, 23, 30, 50, 70):
    print(f"{people:>3} people: P(at least two share a birthday) = {p_shared_birthday(people):.4f}")

def simulate_birthdays(n, trials=20_000):
    days = rng.integers(0, 365, size=(trials, n))
    return sum(len(set(row)) < n for row in days) / trials

print("simulation for 23 people:", simulate_birthdays(23))

In [ ]:
#counting
print("5!             :", math.factorial(5))                  # 120 orderings of 5 things
print("permutations   :", math.perm(5, 2))                    # 20: order matters (first and second place)
print("combinations   :", math.comb(5, 2))                    # 10: order does not matter (a committee of 2)
print("lottery, 6 of 49:", f"{math.comb(49, 6):,}", "-> chance of winning = 1 in", f"{math.comb(49, 6):,}")
print("5-card hands   :", f"{math.comb(52, 5):,}")
print("royal flush    :", f"{4 / math.comb(52, 5):.2e}")

# Check combinations by listing them
from itertools import combinations, permutations
print(len(list(combinations("ABCDE", 2))), len(list(permutations("ABCDE", 2))))

In [ ]:
#shrinking the sample space
def cond_prob(a, b, space):
    """P(A | B): keep only the outcomes where B happened, then ask how many of those are A."""
    given_b = [o for o in space if b(o)]
    if not given_b:
        raise ValueError("P(B) = 0, so P(A | B) is undefined")
    return Fraction(sum(1 for o in given_b if a(o)), len(given_b))

print("P(sum is 8 | first die is 3)      :", cond_prob(lambda o: sum(o) == 8, lambda o: o[0] == 3, two_dice))
print("P(sum is 8)                       :", prob(lambda o: sum(o) == 8, two_dice))
print("P(sum >= 10 | at least one six)   :", cond_prob(lambda o: sum(o) >= 10, lambda o: 6 in o, two_dice))
print("P(heart | red card)               :", cond_prob(heart, lambda c: c[1] in ("hearts", "diamonds"), deck))
print("P(king | face card)               :", cond_prob(king, lambda c: c[0] in ("J", "Q", "K"), deck))
print("P(face card | king)               :", cond_prob(lambda c: c[0] in ("J", "Q", "K"), king, deck))

In [ ]:
#a two-way table
table = pd.DataFrame({"satisfied": [70, 50], "not satisfied": [30, 50]}, index=["remote", "office"])
table.loc["total"] = table.sum()
table["total"] = table.sum(axis=1)
print(table, "\n")

N = table.loc["total", "total"]
p_sat = table.loc["total", "satisfied"] / N
p_sat_remote = table.loc["remote", "satisfied"] / table.loc["remote", "total"]
p_sat_office = table.loc["office", "satisfied"] / table.loc["office", "total"]
p_remote_sat = table.loc["remote", "satisfied"] / table.loc["total", "satisfied"]

print(f"P(satisfied)             = {p_sat:.3f}")
print(f"P(satisfied | remote)    = {p_sat_remote:.3f}")
print(f"P(satisfied | office)    = {p_sat_office:.3f}")
print(f"P(remote | satisfied)    = {p_remote_sat:.3f}    (not the same as P(satisfied | remote))")
print("satisfaction depends on the work mode:", p_sat != p_sat_remote)

In [ ]:
#the multiplication rule, with and without replacement

#The rule: P(A and B) = P(A) × P(B | A).

# Two aces in a row
without_replacement = Fraction(4, 52) * Fraction(3, 51)       # the second draw sees one ace fewer
with_replacement = Fraction(4, 52) * Fraction(4, 52)
print("without replacement:", without_replacement, "=", round(float(without_replacement), 5))
print("with replacement   :", with_replacement, "=", round(float(with_replacement), 5))

# A bag with 5 red and 3 blue marbles, two draws without replacement
red, blue = 5, 3
total = red + blue
p_rr = Fraction(red, total) * Fraction(red - 1, total - 1)
p_bb = Fraction(blue, total) * Fraction(blue - 1, total - 1)
p_mixed = Fraction(red, total) * Fraction(blue, total - 1) + Fraction(blue, total) * Fraction(red, total - 1)
print("both red:", p_rr, "| both blue:", p_bb, "| one of each:", p_mixed, "| total:", p_rr + p_bb + p_mixed)

In [ ]:
#a subtle case, “at least one”
families = list(product("BG", repeat=2))                       # (older child, younger child)
print(families)
print("P(two girls | at least one girl)  :", cond_prob(lambda f: f == ("G", "G"), lambda f: "G" in f, families))
print("P(two girls | the older is a girl):", cond_prob(lambda f: f == ("G", "G"), lambda f: f[0] == "G", families))

In [ ]:
#testing independence
def independent(a, b, space):
    return prob(lambda o: a(o) and b(o), space) == prob(a, space) * prob(b, space)

first_even = lambda o: o[0] % 2 == 0
sum_is_7 = lambda o: sum(o) == 7
sum_is_8 = lambda o: sum(o) == 8

print("first even, sum 7 :", independent(first_even, sum_is_7, two_dice),
      prob(lambda o: first_even(o) and sum_is_7(o), two_dice), "vs", prob(first_even, two_dice) * prob(sum_is_7, two_dice))
print("first even, sum 8 :", independent(first_even, sum_is_8, two_dice),
      prob(lambda o: first_even(o) and sum_is_8(o), two_dice), "vs", prob(first_even, two_dice) * prob(sum_is_8, two_dice))
print("sum 7, sum 8      :", independent(sum_is_7, sum_is_8, two_dice), "(mutually exclusive)")
print("heart, king       :", independent(heart, king, deck))

In [ ]:
#the medical test (the base rate matters)
def bayes(prior, sensitivity, false_positive_rate):
    """P(condition | positive test)."""
    p_positive = prior * sensitivity + (1 - prior) * false_positive_rate
    return prior * sensitivity / p_positive

p1 = bayes(0.01, 0.95, 0.05)
print(f"1% prevalence, 95% sensitivity, 5% false positives -> P(disease | positive) = {p1:.3f}")

# The same thing in natural frequencies: picture 10,000 people
sick = 10_000 * 0.01
healthy = 10_000 - sick
true_positives = sick * 0.95
false_positives = healthy * 0.05
print(f"of {sick:.0f} sick people, {true_positives:.0f} test positive; of {healthy:.0f} healthy people, {false_positives:.0f} test positive")
print(f"so {true_positives:.0f} of {true_positives + false_positives:.0f} positives are really sick: {true_positives / (true_positives + false_positives):.3f}")

# A second, independent positive test: yesterday's posterior becomes today's prior
p2 = bayes(p1, 0.95, 0.05)
print(f"after a second positive test: {p2:.3f}")

In [ ]:
#how the base rate drives the answer
prevalence = np.linspace(0.001, 0.5, 200)
posterior = bayes(prevalence, 0.95, 0.05)
for p in (0.001, 0.01, 0.10, 0.50):
    print(f"prevalence {p:>5.3f} -> P(disease | positive) = {bayes(p, 0.95, 0.05):.3f}")

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(prevalence, posterior)
ax.set_xlabel("prevalence (the prior)")
ax.set_ylabel("P(disease | positive)")
plt.show()

In [ ]:
#the prosecutor’s fallacy
def p_source_given_match(suspects, random_match_probability):
    """Assume a uniform prior over the suspects, the true source always matches, and the others match by chance."""
    expected_innocent_matches = (suspects - 1) * random_match_probability
    return 1 / (1 + expected_innocent_matches)

rmp = 1 / 1_000_000                                         # "only 1 in a million people would match by chance"
for suspects in (5_000_000, 100_000, 1_000):
    print(f"{suspects:>9,} possible sources -> P(source | match) = {p_source_given_match(suspects, rmp):.4f}")

In [ ]:
#the Monty Hall problem, by simulation
def monty_hall(switch, trials=50_000):
    wins = 0
    for _ in range(trials):
        car = rng.integers(0, 3)
        pick = rng.integers(0, 3)
        options = [d for d in range(3) if d != pick and d != car]      # the host opens a goat door, never yours, never the car
        opened = rng.choice(options)
        if switch:
            pick = next(d for d in range(3) if d != pick and d != opened)
        wins += pick == car
    return wins / trials

print("stay  :", monty_hall(switch=False))                   # about 1/3
print("switch:", monty_hall(switch=True))                    # about 2/3

In [ ]:
#a simulated conditional probability
rolls = rng.integers(1, 7, size=(1_000_000, 2))              # 1 to 6: the upper bound 7 is NOT included
has_six = (rolls == 6).any(axis=1)
big_sum = rolls.sum(axis=1) >= 10

print("exact     P(sum >= 10 | at least one six):", float(Fraction(5, 11)))
print("simulated P(sum >= 10 | at least one six):", round((big_sum & has_six).sum() / has_six.sum(), 4))
print("simulated P(at least one six):", round(has_six.mean(), 4), "exact:", round(11 / 36, 4))

In [ ]:
#a helper that we can reuse
def simulate_probability(experiment, trials=100_000):
    """Run a function that returns True/False many times and return the fraction of Trues."""
    return sum(bool(experiment()) for _ in range(trials)) / trials

def three_coins_two_heads():
    return rng.integers(0, 2, size=3).sum() == 2

def draw_two_aces():
    first, second = rng.choice(52, size=2, replace=False)     # card ids 0 to 51; ids 0 to 3 are the aces
    return first < 4 and second < 4

print("exactly 2 heads in 3 flips:", simulate_probability(three_coins_two_heads), "(exact 3/8 = 0.375)")
print("two aces in a row         :", simulate_probability(draw_two_aces, 200_000), f"(exact 1/221 = {1 / 221:.4f})")